# Module 08 AI Workbook — solution and explanation

> Try the exercise yourself before reading this.

## The adversarial bug

[../adversarial_nbody_buggy.cu](./adversarial_nbody_buggy.cu) computes forces
and integrates positions in **one** kernel:

```cpp
// inside force_and_integrate, one thread per body i:
p[i].vx += dt*Fx; ...          // update velocity
p[i].x  += p[i].vx*dt; ...      // AND advance position — in the same kernel
```

Every thread `i` reads `p[j]` for all `j` to compute its force. But thread `j` is
concurrently writing `p[j].x/y/z` at the bottom of the same kernel. There is no
ordering between different threads (or different blocks) within a single launch,
so thread `i` may read some positions that have already advanced by one step and
others that have not. The physics becomes a nondeterministic mix of two time
steps.

**Why it can look fine (or even "better"):** the trajectory of body 0 may still
look plausible, and because the fused kernel skips a whole synchronized phase,
the reported "interactions/sec" can be *higher*. A performance number without a
correctness gate is worthless.

## The fix

Separate the two phases into **two kernels** with a device synchronization
between them, so all forces are computed from the old positions before any
position moves. See [nbody_solution.cu](solutions/nbody_solution.cu):

```cpp
bodyForce<<<blocks, threads>>>(p, dt, n);   // writes only velocity
cudaDeviceSynchronize();                     // global barrier between phases
integrate<<<blocks, threads>>>(p, dt, n);   // then advance positions
cudaDeviceSynchronize();
```

`bodyForce` writes only `p[i].v*` (each thread writes its own body, reads only
positions), so it has no read/write conflict; `integrate` then updates positions
once all forces exist.

## The performance angle (Phase 5)

All-pairs N-body is O(N²) arithmetic over O(N) data — **compute-bound**, which is
where GPUs shine. The next optimization is **shared-memory tiling**: each block
loads a tile of body positions into shared memory, and all threads in the block
reuse it, cutting global-memory traffic. Predict the speedup from the ratio of
global loads before vs after tiling, then confirm with the profiler.

## Mapping to the 5-phase loop

- **PREDICT:** flag the fused phases as a whole-grid race before running.
- **VERIFY:** compare final positions against the CPU reference over all bodies —
  the fused version fails this even though it prints a plausible body 0.
- **DIAGNOSE:** confirm compute-bound behaviour, then justify shared-memory
  tiling as the next step with profiler evidence.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!nvcc -O3 -arch=native solutions/nbody_solution.cu -o sol && ./sol